In [3]:
x = "test"

In [6]:
import base64
import hashlib
import hmac
import uuid
import pandas as pd
from datetime import datetime

import requests


def sign_request(method, pathAndQuery, body, apiKey, secret):
    verb = method.upper()
    utc_now = datetime.utcnow().strftime("%a, %d %b %Y %H:%M:%S ") + "GMT"
    nonce = str(uuid.uuid4())

    content_digest = hashlib.sha256(str(body).encode("utf-8")).digest()
    content_hash = base64.b64encode(content_digest).decode("utf-8")

    string_to_sign = apiKey + ";" + pathAndQuery + ";" + verb + ";" + utc_now + ";" + nonce + ";" + content_hash

    decoded_secret = secret.encode("utf-8")
    digest = hmac.new(decoded_secret, str(string_to_sign).encode("utf-8"), hashlib.sha256).digest()

    signature = base64.b64encode(digest).decode("utf-8")

    return {
        "x-date": utc_now,
        "x-hmac256-signature": apiKey + ";" + nonce + ";" + signature,
        "x-content-sha256": content_hash,
    }

def get_unrealized_tax_lot(base_url, api_key, api_secret, nav_global_fund_id, report_date):
    method = "GET"
    #path = "/navapigateway/api/v1/" + api_root + "/" + api_call
    path = "/navapigateway/api/v1/FundReportData/GetUnRealizedTaxLotForFund?globalFundID=" + nav_global_fund_id + "&reportDate=" + report_date
    url = base_url + path

    body = ""  # GET: empty string

    headers = sign_request(method, path, body, api_key, api_secret)
    headers["Accept"] = "application/json"

    # Some gateways also require this even if docs don't; harmless to include:
    headers["Authorization"] = headers["x-hmac256-signature"]

    resp = requests.get(url, headers=headers, timeout=(10, 60))
    if not resp.ok:
        print("URL:", url)
        print("Status:", resp.status_code)
        print("Resp text:", resp.text[:500])
        print("x-date:", headers["x-date"])
        print("x-content-sha256:", headers["x-content-sha256"])
        print("x-hmac256-signature:", headers["x-hmac256-signature"])
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:1500]}")

    return pd.DataFrame(resp.json())

BASE_URL = "https://api.navfundservices.com"
API_KEY = "API-NT9KQUAP"
API_SECRET = "JU5vRHRcBvgHYAJXmHSvsB2zY/aqDHGMDJoNI2qVIIA="

nav_unrlz_view = get_unrealized_tax_lot(BASE_URL, API_KEY, API_SECRET, "175076", "01-30-2026")

In [8]:
    df_unrealized_market_value = pd.DataFrame({

        "Fund Code": f"INDU",
        "Value Date": '2026-01-30',
        "Identifier": nav_unrlz_view["ISIN"],
        "Identifier 2": nav_unrlz_view["Ticker"],
        "Description": nav_unrlz_view["IssuerName"],
        "Currency": nav_unrlz_view["Currency"],
        "Qty": nav_unrlz_view["OpenQuantity"],
        "Market Value (Base)": nav_unrlz_view["MarketValueBase"]
 

    })

In [14]:
df_unrealized_market_value_grouped = (
    df_unrealized_market_value
    .groupby(["Fund Code", "Value Date", "Identifier", "Identifier 2", "Description", "Currency"], as_index=False)
    .agg(**{
        "Qty": ("Qty", "sum"),
        "Market Value (Base)": ("Market Value (Base)", "sum")
    })


)